# Objectives
Welcome to the fifth Natural Language Processing lab.

Today's objectives are to:

1. Classify text using the **linear models** of Naive Bayes, Logistic Regression, and Linear SVM.
2. Observe how different **preprocessing and representation options** affect classification performance.
3. Inspect **word-feature model weights** for interpretation of results.
4. Explore classification **performance metrics** and cost trade-offs.

# Notebook Setup

## Packages

In [1]:
#| code-summary: 'Import packages'
#| output: false
# Dataset packages
from datasets import load_dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter
from imblearn.under_sampling import RandomUnderSampler

# Data packages
import pandas as pd
pd.set_option('display.max_colwidth', None) #default: 50 chars
pd.set_option('display.max_columns', None) #default: 20 columns
import numpy as np

# NLP packages
import spacy
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Classification libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import make_scorer, f1_score, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Datasets

In [2]:
#| code-summary: 'Dataset #1: Spam Detection'
ds = load_dataset('ucirvine/sms_spam')
df = ds['train'].to_pandas()
df = df.rename(columns={'sms': 'text', 'label': 'y'})
df.sample(2)

c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Serenity\.cache\huggingface\hub\datasets--ucirvine--sms_spam. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\huggingface_hub\file_download.py

,text,y
4387,", im .. On the snowboarding trip. I was wondering if your planning to get everyone together befor we go..a meet and greet kind of affair? Cheers, \n",0
1589,At 4. Let's go to bill millers\n,0


In [9]:
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "crowdflower/twitter-airline-sentiment/versions/4",
    "Tweets.csv",
    pandas_kwargs={"encoding": "latin-1"},
)

100%|██████████| 1.08M/1.08M [00:00<00:00, 4.31MB/s]


ValueError: Error reading file: Error tokenizing data. C error: Expected 1 fields in line 3, saw 4


In [10]:
df = df.rename(columns={"airline_sentiment": "y"})
df.sample(2)

,text,y
5103,Nope thats fine. I might have a nap tho! \n,0
2371,"That day ü say ü cut ur hair at paragon, is it called hair sense? Do ü noe how much is a hair cut? \n",0


In [11]:
#| code-summary: 'Dataset #3: Authorship Detection'
ds = load_dataset('CogniSAL/MDAIGT')
df = pd.concat([ds['train_news'].to_pandas(), ds['train_news'].to_pandas()])
df = df.rename(columns={'label': 'y'})
df = df.sample(frac=0.1, random_state=42) # reduce for vectorizing time
df.sample(2)

c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Serenity\.cache\huggingface\hub\datasets--CogniSAL--MDAIGT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train_academic_articles split: 100%|██████████| 10000/10000 [00:00<00:00, 43582.94 exampl

,id,text,y
8197,8198,"Biker Robert Reed and His Canine Companions A Journey Across America Robert Reed, an avid motorcycle enthusiast from Pennsylvania, embarked on an unprecedented adventure over two years ago when he set out to traverse the United States with his two loyal dogs, Andy and Maddie. Dressed in miniature leather jackets and goggles, the duo rode sidecar with Reed, experiencing the country's breathtaking landscapes and historic landmarks. This unique trio, often referred to as ""The Biking Bulldogs,"" has since visited 22 states and covered approximately 15,000 miles. Their journey has taken them from the sunny beaches of California to the majestic Rocky Mountains, the iconic Gateway Arch in St. Louis, and the beautiful vineyards of Napa Valley, among other memorable destinations. Andy, a 7-year-old English bulldog with a regal demeanor and distinctive wrinkled features, and Maddie, a lively 5-year-old English bulldog with boundless energy, are seldom seen without their distinctive outfits. These protective garments, custom-made by a local seamstress, shield them from the elements, ensuring they remain comfortable and content throughout their travels. ""I always wanted to see the country, and I knew I couldn't do it alone,"" Reed shared. ""Having my dogs with me makes every mile worthwhile. They're not just my companions, but they're my family."" Reed, a retired firefighter, had previously taken shorter motorcycle trips with his late wife, who passed away due to an illness. Struggling with the grief, he decided to take on this ambitious journey to rediscover the joy of exploration and to honor his wife's memory. As they continued their travels, Reed documented their experiences on social media, amassing a devoted following of over 100,000 people. Their heartwarming stories of adventure, love, and loyalty have inspired countless individuals and touched the lives of many. ""The Biking Bulldogs"" have also used their platform to promote animal welfare and adoption, which is close to Reed's heart. He often shares stories of other rescued animals he's encountered on his travels and encourages his followers to adopt from shelters. As Reed and his faithful companions prepare for the final leg of their journey, they look forward to visiting the remaining states and making new memories. Their epic adventure serves as a testament to the indomitable human spirit and the unbreakable bond between man and man's best friend.",machine
6775,6776,"(CNET) -- Coulomb Technologies on Wednesday said it will build 4,600 electric vehicle charging stations in nine regions of the U.S., funded by $37 million in grants. More than half of the charging stations in the program called ChargePoint America will go in public locations, such as overnight parking spaces, offices, or retail locations. About 2,000 going into people's homes as part of the purchase of an electric vehicle. The goal of the charging network and others like it is to make electric vehicles more accessible to consumers, as these cars come to market in the coming months, said Richard Lowenthal, the CEO of Campbell, California-based Coulomb Technologies during a press conference. The program differs from other electric vehicle charging networks in that Coulomb is working with Ford Motor, General Motors, and Smart USA on the roll out, Lowenthal said. Ford will release an electric Transit Connect utility van and electric Ford Focus sedan in 2011, GM plans to release the Volt by November, and the electric Smart fortwo will begin pilot testing in the U.S. by 2011. The stations themselves will be equipped with 220-volt service at either 16 or 30 amps, which enables faster charging than regular home electrical service. Lowenthal said that an electric car with a large battery could charge from empty in under four hours. Drivers will be able to find available charging stations using either an iPhone or Blackberry application, which can also send alerts if a car battery is running low or

In [12]:
#| code-summary: 'Train-Test Split and class balancing'
RANDOM_SEED = 42
df_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_SEED)
print(df_train['y'].value_counts())
rus = RandomUnderSampler(random_state=42)
df_train_bal = pd.DataFrame()
df_train_bal['text'], df_train_bal['y'] = rus.fit_resample(df_train['text'].to_frame(), df_train['y'])
df_train_bal['y'].value_counts()

y
human      800
machine    800
Name: count, dtype: int64


y
human      800
machine    800
Name: count, dtype: int64

# Preprocessing

In [13]:
#| code-summary: 'Preprocessing from Lab 2'
nlp = spacy.load('en_core_web_sm')
def spacy_tokenizer(text):
    '''Tokenizes and normalizes one string of text'''
    doc = nlp(text) 
    # MODIFY BELOW: lower_, is_alpha, lemma_, is_stop (normalization from Lab 2)
    tokens = [token.text for token in doc if token.is_alpha]
    return tokens

# Pre-check test entry with all OOV else get_mean_vector error
tokens = df_train_bal['text'].apply(spacy_tokenizer)
df_train_bal = df_train_bal[tokens.str.len() > 0].copy()
tokens = df_test['text'].apply(spacy_tokenizer)
df_test = df_test[tokens.str.len() > 0].copy()

# Representation

## Count/TF-IDF

In [14]:
#| code-summary: 'Bag of Words from Lab 3'
bow_vectorizer = CountVectorizer(
    tokenizer = spacy_tokenizer,
    token_pattern = None,
    lowercase = False,
    min_df = 2
)
X_train_bow = bow_vectorizer.fit_transform(df_train_bal['text'])
X_test_bow = bow_vectorizer.transform(df_test['text'])
print(f'The vocabulary size is {len(bow_vectorizer.vocabulary_)} tokens\n')
pd.DataFrame(X_train_bow.toarray(), index=df_train_bal.index, columns=bow_vectorizer.get_feature_names_out()).head()

The vocabulary size is 15509 tokens



,A,ABC,AC,ACA,ACLU,AFP,AG,AIDS,AIDs,AM,AMIA,AP,AQAP,ATM,ATOL,ATP,Aaron,Aasiya,Abaaoud,Abbey,Abdel,Abdelhamid,Abdul,AbdulMutallab,Abdulaziz,Abdullah,Abe,Abed,Aberdeen,Abhishek,About,Above,Abraham,Abrahams,Abrams,Abu,Abzug,Academy,Accord,According,Accra,Accused,Achieve,Achievement,Across,Act,Acting,Action,Actions,Active,Activist,Activities,Actor,Actors,Actress,Actually,Adam,Adding,Additional,Additionally,Adele,Adidas,Administration,Administrator,Adolf,Adrian,Advanced,Adventures,Advocates,Affairs,Affleck,Affordable,Afghan,Afghanistan,Africa,African,After,Afterwards,Again,Against,Agatha,Age,Agency,Agent,Agree,Agriculture,Ah,Ahlers,Ahmed,Aid,Air,AirAsia,Airbus,Aires,Airline,Airlines,Airplane,Airport,Airways,Akaichi,Akka,Al,Alabama,Alain,Alamuddin,Alan,Alaska,Alaskan,Alba,Albert,Alberto,Albie,Album,Alcohol,Alec,Alejandro,Aleppo,Alert,Alex,Alexa,Alexander,Alexandre,Alexandria,Alexis,Alfie,Algeria,Ali,Alice,Alison,Alito,Alive,All,Allegations,Allen,Alliance,Allred,Almost,Along,Alongside,Alonso,Alpine,Also,Alternatives,Although,Always,Alzheimer,Amal,Amanda,Amazon,Amber,Ambrose,Amedy,Amelia,Amendment,America,American,Americans,Americas,Amid,Amidst,Among,Amphipolis,Amsterdam,Amy,An,Ana,Anadappa,Analysis,Analysts,Analytics,Anchorage,And,Anderlecht,Anders,Andersen,Anderson,Andre,Andrea,Andrew,Android,Andy,Anfield,Angel,Angela,Angeles,Angelina,Angry,Angus,Animal,Aniston,Ankara,Ann,Anna,Annapolis,Anne,Annie,Announces,Annual,Another,Antarctic,Anthony,Anti,Antoine,Antonio,Any,Anybody,Anyone,Anytime,Apollo,App,Apparel,Apparently,Appeals,Apple,Applied,Apps,April,Apsley,Arab,Arabia,Arabian,Arch,Archaeologists,Archer,Archives,Arctic,Are,Aren,Arena,Argentina,Argentine,Argo,Arizona,Arkansas,Arlington,Armed,Army,Arnold,Around,Arpaio,Arquette,Arrest,Arrested,Arsenal,Arsene,Art,Arteta,Arthur,Article,Artists,Arts,Arturo,Artwalk,As,Asafa,Ascot,Ashdown,Ashley,Ashton,Ashwin,Asia,Asian,Asked,Aspinal,Assad,Assembly,Assistance,Assistant,Assisted,Associated,Association,Aston,Astronomers,At,Athay,Athens,Athletic,Atkinson,Atlanta,Atlantic,Atlantis,Atletico,Attorney,Attorneys,Aubameyang,Auckland,Auction,August,Aunt,Aurora,Austin,Australia,Australian,Austria,Austrian,Authorities,Authority,Autumn,Available,Avengers,Aviation,Aviv,Award,Awards,Ayatollah,Ayliffe,Aylward,Azamat,Azarenka,B,BA,BBC,BC,BFW,BJP,BMI,BMW,BP,BRICS,Babies,Baby,Baca,Bach,Bachelor,Back,Backstreet,Bad,Baghdad,Bahrain,Bailey,Baines,Bakambu,Baker,Baldwin,Bale,Bali,Ball,Ballmer,Balotelli,Balpa,Baltimore,Band,Bang,Bangerz,Bangkok,Bangladesh,Bank,Banks,Baptist,Barack,Barbara,Barbie,Barcelona,Barlow,Barnard,Barnes,Barranco,Barrett,Barry,Bartlett,Base,Based,Basel,Bashar,Basically,Basin,Bass,Bastian,Bastille,Bata,Bath,Batman,Battle,Battleship,Baucus,Bay,Bayern,Be,Beach,Bear,Beard,Beatrice,Beats,Beauty,Beaver,Because,Beck,Beckham,Becky,Bedfordshire,Beef,Before,Behring,Beijing,Being,Belgian,Belgium,Belinda,Bell,Bella,Bellevue,Below,Ben,Benedict,Bengaluru,Benghazi,Benjamin,Benson,Beracasa,Berdych,Berkeley,Berlin,Bermuda,Bernanke,Bernd,Bernie,Berrer,Berry,Besic,Besides,Best,Beth,Betty,Between,Beyonce,Beyoncé,Beyond,Bharatiya,Bialek,Bianchi,Bible,Biden,Bieber,Biel,Big,Bike,Biker,Bilbao,Bill,Billboard,Billy,Bin,Biological,Biologists,Biology,Bird,Birmingham,Bishop,Bismarck,Black,BlackBerry,Blackberry,Blair,Blake,Blasio,Blatter,Blind,Blizzard,Blogger,Blood,Bloomberg,Blue,Blues,Bluetooth,Bluey,Board,Bob,Bobby,Body,Boehner,Boeing,Boiled,Boivin,Bokila,Boko,Bold,Bollinger,Bologna,Bomb,Bombings,Bon,Bonanno,Bonas,Bond,Bone,Bony,Book,Boot,Boots,Border,Borders,Borges,Borini,Boris,Born,Borno,Bosnian,Boston,Botelho,Both,Boulder,Boullier,Boumeddiene,Bourdain,Bowl,Box,Boxing,Boy,Boyd,Boys,Bozek,Brad,Bradley,Brady,Braille,Brain,Braley,Brand,Brando,Brandon,Bravo,Brazil,Brazilian,Break,Breakfast,Breaking,Breast,Breivik,Brendan,Brennan,Brian,Bridge,Bridgeport,Bridges,Bridget,Bridgewater,Briers,Brighton,Brigid,Bristol,Britain,British,Britney,Britons,Brits,Brittany,Broadway,Brogues,Brom,Bromwich,Bronx,Bronze,Brooke,Brooklyn,Brother

In [15]:
#| code-summary: 'TF-IDF from Lab 3'
tfidf_vectorizer = TfidfVectorizer(
    tokenizer = spacy_tokenizer,
    token_pattern = None,
    lowercase = False,
    min_df = 2
)
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train_bal['text'])
X_test_tfidf = tfidf_vectorizer.transform(df_test['text'])
print(f'The vocabulary size is {len(tfidf_vectorizer.vocabulary_)} tokens\n')
pd.DataFrame(X_train_tfidf.toarray(), index=df_train_bal.index, columns=tfidf_vectorizer.get_feature_names_out()).head()

The vocabulary size is 15509 tokens



,A,ABC,AC,ACA,ACLU,AFP,AG,AIDS,AIDs,AM,AMIA,AP,AQAP,ATM,ATOL,ATP,Aaron,Aasiya,Abaaoud,Abbey,Abdel,Abdelhamid,Abdul,AbdulMutallab,Abdulaziz,Abdullah,Abe,Abed,Aberdeen,Abhishek,About,Above,Abraham,Abrahams,Abrams,Abu,Abzug,Academy,Accord,According,Accra,Accused,Achieve,Achievement,Across,Act,Acting,Action,Actions,Active,Activist,Activities,Actor,Actors,Actress,Actually,Adam,Adding,Additional,Additionally,Adele,Adidas,Administration,Administrator,Adolf,Adrian,Advanced,Adventures,Advocates,Affairs,Affleck,Affordable,Afghan,Afghanistan,Africa,African,After,Afterwards,Again,Against,Agatha,Age,Agency,Agent,Agree,Agriculture,Ah,Ahlers,Ahmed,Aid,Air,AirAsia,Airbus,Aires,Airline,Airlines,Airplane,Airport,Airways,Akaichi,Akka,Al,Alabama,Alain,Alamuddin,Alan,Alaska,Alaskan,Alba,Albert,Alberto,Albie,Album,Alcohol,Alec,Alejandro,Aleppo,Alert,Alex,Alexa,Alexander,Alexandre,Alexandria,Alexis,Alfie,Algeria,Ali,Alice,Alison,Alito,Alive,All,Allegations,Allen,Alliance,Allred,Almost,Along,Alongside,Alonso,Alpine,Also,Alternatives,Although,Always,Alzheimer,Amal,Amanda,Amazon,Amber,Ambrose,Amedy,Amelia,Amendment,America,American,Americans,Americas,Amid,Amidst,Among,Amphipolis,Amsterdam,Amy,An,Ana,Anadappa,Analysis,Analysts,Analytics,Anchorage,And,Anderlecht,Anders,Andersen,Anderson,Andre,Andrea,Andrew,Android,Andy,Anfield,Angel,Angela,Angeles,Angelina,Angry,Angus,Animal,Aniston,Ankara,Ann,Anna,Annapolis,Anne,Annie,Announces,Annual,Another,Antarctic,Anthony,Anti,Antoine,Antonio,Any,Anybody,Anyone,Anytime,Apollo,App,Apparel,Apparently,Appeals,Apple,Applied,Apps,April,Apsley,Arab,Arabia,Arabian,Arch,Archaeologists,Archer,Archives,Arctic,Are,Aren,Arena,Argentina,Argentine,Argo,Arizona,Arkansas,Arlington,Armed,Army,Arnold,Around,Arpaio,Arquette,Arrest,Arrested,Arsenal,Arsene,Art,Arteta,Arthur,Article,Artists,Arts,Arturo,Artwalk,As,Asafa,Ascot,Ashdown,Ashley,Ashton,Ashwin,Asia,Asian,Asked,Aspinal,Assad,Assembly,Assistance,Assistant,Assisted,Associated,Association,Aston,Astronomers,At,Athay,Athens,Athletic,Atkinson,Atlanta,Atlantic,Atlantis,Atletico,Attorney,Attorneys,Aubameyang,Auckland,Auction,August,Aunt,Aurora,Austin,Australia,Australian,Austria,Austrian,Authorities,Authority,Autumn,Available,Avengers,Aviation,Aviv,Award,Awards,Ayatollah,Ayliffe,Aylward,Azamat,Azarenka,B,BA,BBC,BC,BFW,BJP,BMI,BMW,BP,BRICS,Babies,Baby,Baca,Bach,Bachelor,Back,Backstreet,Bad,Baghdad,Bahrain,Bailey,Baines,Bakambu,Baker,Baldwin,Bale,Bali,Ball,Ballmer,Balotelli,Balpa,Baltimore,Band,Bang,Bangerz,Bangkok,Bangladesh,Bank,Banks,Baptist,Barack,Barbara,Barbie,Barcelona,Barlow,Barnard,Barnes,Barranco,Barrett,Barry,Bartlett,Base,Based,Basel,Bashar,Basically,Basin,Bass,Bastian,Bastille,Bata,Bath,Batman,Battle,Battleship,Baucus,Bay,Bayern,Be,Beach,Bear,Beard,Beatrice,Beats,Beauty,Beaver,Because,Beck,Beckham,Becky,Bedfordshire,Beef,Before,Behring,Beijing,Being,Belgian,Belgium,Belinda,Bell,Bella,Bellevue,Below,Ben,Benedict,Bengaluru,Benghazi,Benjamin,Benson,Beracasa,Berdych,Berkeley,Berlin,Bermuda,Bernanke,Bernd,Bernie,Berrer,Berry,Besic,Besides,Best,Beth,Betty,Between,Beyonce,Beyoncé,Beyond,Bharatiya,Bialek,Bianchi,Bible,Biden,Bieber,Biel,Big,Bike,Biker,Bilbao,Bill,Billboard,Billy,Bin,Biological,Biologists,Biology,Bird,Birmingham,Bishop,Bismarck,Black,BlackBerry,Blackberry,Blair,Blake,Blasio,Blatter,Blind,Blizzard,Blogger,Blood,Bloomberg,Blue,Blues,Bluetooth,Bluey,Board,Bob,Bobby,Body,Boehner,Boeing,Boiled,Boivin,Bokila,Boko,Bold,Bollinger,Bologna,Bomb,Bombings,Bon,Bonanno,Bonas,Bond,Bone,Bony,Book,Boot,Boots,Border,Borders,Borges,Borini,Boris,Born,Borno,Bosnian,Boston,Botelho,Both,Boulder,Boullier,Boumeddiene,Bourdain,Bowl,Box,Boxing,Boy,Boyd,Boys,Bozek,Brad,Bradley,Brady,Braille,Brain,Braley,Brand,Brando,Brandon,Bravo,Brazil,Brazilian,Break,Breakfast,Breaking,Breast,Breivik,Brendan,Brennan,Brian,Bridge,Bridgeport,Bridges,Bridget,Bridgewater,Briers,Brighton,Brigid,Bristol,Britain,British,Britney,Britons,Brits,Brittany,Broadway,Brogues,Brom,Bromwich,Bronx,Bronze,Brooke,Brooklyn,Brother

## Embeddings

In [16]:
#| code-summary: 'Word2Vec from Lab 4'
# Download to FILE (1662.8MB, ~30 minutes) once (else just returns downloaded file path)
wv_path = api.load('word2vec-google-news-300', return_path=True)

# Load vectors into RAM (limit if needed)
limit = 1000000
wv_vectors = KeyedVectors.load_word2vec_format(wv_path, binary=True, limit=limit)
print(f'Loaded KeyedVectors with {len(wv_vectors)} vectors in {wv_vectors.vector_size} dimensions')

#| code-summary: 'Generate mean embedding vectors for each document'
X_train_mean_wv = np.vstack([wv_vectors.get_mean_vector(spacy_tokenizer(doc)).reshape(1, -1) for doc in df_train_bal['text']])
X_test_mean_wv = np.vstack([wv_vectors.get_mean_vector(spacy_tokenizer(doc)).reshape(1, -1) for doc in df_test['text']])

pd.DataFrame(X_train_mean_wv, index=df_train_bal.index).head()

Loaded KeyedVectors with 1000000 vectors in 300 dimensions


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299
5903,0.014426,0.014005,0.021852,0.018497,0.005867,-0.033191,0.002082,-0.042596,0.032506,0.045204,-0.019508,-0.026656,0.006145,0.021831,-0.034212,0.003694,0.023187,0.023144,0.007048,-0.012668,-0.009819,0.026328,-0.017451,-0.011728,0.027078,-0.011569,-0.031783,0.028818,0.016716,0.018335,-0.005151,-0.009505,-0.019306,-0.016954,0.003485,-0.005387,-0.014614,-0.011707,0.010326,0.026830,0.032739,-0.031915,0.038083,-0.009990,-0.000510,-0.005135,-0.007729,0.004062,0.027145,0.032266,0.006575,0.031737,0.003735,-0.010632,-0.027775,0.019339,-0.018524,-0.023141,0.010547,-0.026340,-0.029262,0.016730,-0.030493,-0.063066,-0.022877,0.014589,0.015547,0.028707,0.002786,0.044673,0.013731,-0.010875,0.023319,0.017266,-0.042808,-0.026057,0.029462,0.035110,0.008527,0.036385,-0.000799,-0.040883,0.017831,-0.001171,-0.006067,-0.025726,-0.045084,0.038697,-0.011285,0.000181,0.017849,0.012940,-0.017890,-0.038520,-0.018079,-0.027136,0.011116,-0.008501,-0.027135,0.007881,-0.015753,-0.022809,0.003651,-0.004814,-0.011045,-0.022476,-0.004201,-0.014369,-0.007125,-0.012333,0.008390,-0.002848,-0.009220,-0.009538,0.019483,0.012655,0.026784,0.003458,0.043459,0.027280,-0.052663,-0.001955,-0.016958,0.019783,-0.015575,-0.010773,-0.006793,-0.030684,-0.000763,0.027090,-0.002932,-0.035439,-0.046033,-0.012338,-0.024280,-0.029154,-0.002016,-0.008153,-0.041501,0.023872,0.004038,-0.019929,0.013944,-0.008200,-0.001767,0.030989,-0.012965,-0.022980,-0.029732,0.011730,0.025282,0.014849,-0.026816,0.024361,-0.005660,0.007743,-0.025615,-0.039668,0.001939,-0.014001,-0.022481,0.012005,0.035328,-0.000661,-0.000856,-0.014095,0.025760,-0.008741,0.019454,-0.021059,-0.041087,-0.022813,-0.009596,-0.043330,-0.015533,-0.024522,0.026225,-0.023367,-0.008395,-0.000957,-0.028829,-0.011278,0.000771,0.016053,-0.010349,-0.010768,0.000413,0.013922,0.040837,0.010545,0.010650,-0.003660,0.032277,0.000150,-0.035601,0.019612,0.002364,-0.009782,-0.034149,-0.057675,0.005675,0.012421,-0.006379,-0.001628,0.012040,-0.020268,-0.008528,0.008049,0.008391,-0.018101,-0.009277,0.018175,-0.002994,0.022613,-0.030794,-0.011841,0.027227,-0.019903,-0.055413,-0.010177,0.002210,0.007418,0.012068,-0.027701,0.015725,-0.016505,0.015181,0.013134,0.030976,0.002264,0.001727,-0.023431,0.010055,0.011977,0.008935,-0.001876,0.005330,-0.021825,0.020017,0.021050,0.014117,-0.001838,0.022684,-0.057890,0.003604,0.002881,0.008408,0.017160,0.015179,-0.005272,0.011676,0.004261,0.007951,0.007667,0.011389,0.003355,0.031649,0.004641,-0.023600,-0.019856,0.004509,0.024860,-0.008525,0.022403,0.042191,0.038352,0.011422,0.019658,-0.010986,0.000524,0.017612,0.026855,0.009606,0.004861,0.040389,-0.030201,-0.022472,-0.049524,-0.001976,-0.010071,0.020109,-0.014247,-0.005037,0.030724,0.003610,0.003678,-0.041286,0.003178,-0.008369,0.021877,-0.022679,-0.002873,-0.044184,0.002238,-0.019691,-0.003303,-0.008211,-0.013183,0.016145,-0.019254
9826,0.000289,0.020248,0.014810,0.038037,-0.021074,

In [17]:
#| code-summary: 'Doc2Vec from Lab 4'
# Prepare Doc2Vec-specific format
train_tagged_docs = [TaggedDocument(spacy_tokenizer(doc), [i]) for i, doc in enumerate(df_train_bal['text'])]
test_tagged_docs = [TaggedDocument(spacy_tokenizer(doc), [i]) for i, doc in enumerate(df_test['text'])]

# Configure and TRAIN Doc2Vec model
X_train_dv_vectors = Doc2Vec(train_tagged_docs, vector_size=300, window=2, min_count=2, epochs=40, seed=42)
X_test_dv_vectors = Doc2Vec(test_tagged_docs, vector_size=300, window=2, min_count=2, epochs=40, seed=42)
X_train_dv = list(X_train_dv_vectors.dv.vectors)
X_test_dv = list(X_test_dv_vectors.dv.vectors)

pd.DataFrame(X_train_dv, index=df_train_bal.index).head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299
5903,0.035317,0.320354,-0.926446,0.778507,-0.059755,0.351596,0.315058,0.109383,-0.324056,0.936658,0.734125,0.871844,-0.592808,0.673142,0.609033,0.044005,-1.132675,-0.241563,-0.344555,0.856007,0.708678,-1.025080,0.029638,1.005595,0.361247,0.598653,-0.365538,0.541625,-0.429494,-0.353045,-0.900789,-0.228325,-1.960521,0.330806,0.383495,0.943457,0.420437,-0.285185,-0.743700,-0.375696,0.747443,-0.065201,0.125298,-0.610115,-0.266841,0.936735,-0.462211,-0.553087,0.951625,0.215494,0.299106,-0.287706,1.230846,-0.091514,0.276991,-0.821316,0.640729,0.438131,-0.371039,0.765477,0.031812,0.302615,0.147327,0.945189,0.398307,0.234827,0.562567,-0.203561,-0.140217,-0.037011,-0.281864,-0.235487,-0.508104,-0.624917,0.100906,0.439759,-0.141409,-0.358180,0.379986,-0.176133,-0.843341,0.054825,-0.308751,0.404779,-0.346216,-0.740472,0.031166,-0.039507,-0.226036,0.982705,0.323679,0.541053,-1.103988,-0.579177,-0.126818,-0.179737,-0.112322,0.610579,0.469227,0.170360,-0.943476,0.114566,0.289666,-0.117555,0.709197,0.352069,0.601780,0.341780,-1.231716,0.505367,-0.035452,-0.167084,-0.325946,1.254683,0.369214,0.574016,0.005209,1.287266,0.438763,-0.833610,0.315849,0.893006,1.071789,0.013488,-0.303836,0.382645,-1.517060,-0.083133,0.622597,0.125943,-0.270983,0.132887,0.177424,0.755619,0.337183,1.066224,-0.306400,0.322612,0.191908,-0.500999,0.282045,-0.408476,-0.097679,-0.750726,-0.802931,0.298652,1.372146,0.219836,1.317986,-0.524992,0.261006,0.832178,-0.126161,-1.308103,0.598533,-0.360830,-0.425448,-0.446683,0.367291,-0.267063,1.420334,0.231570,-0.191524,-0.908266,-1.301677,-0.349721,-0.173455,-0.184360,-0.438400,-0.892918,-0.044725,-0.355349,0.145058,-0.003688,0.511485,-0.380218,-0.467191,0.282609,0.267754,0.440329,-0.399576,-0.565247,-0.245982,0.568154,0.755469,-0.138341,0.427678,-1.617154,1.374691,1.015082,0.406480,0.441688,0.546237,-0.504221,-0.894389,-0.035886,-0.139550,-0.441128,0.317120,0.311725,1.361075,-0.742835,-0.087829,-0.936482,0.093331,0.200093,-1.507786,0.504151,0.227964,0.325907,-0.529120,-0.850297,1.126417,-0.915282,-0.346936,-1.020483,-0.397602,0.078668,0.673764,-0.670004,-0.478466,0.225794,1.067223,-0.529117,-0.525636,0.345969,0.945780,0.990379,-0.924881,0.461486,-0.719871,-1.337548,-0.420058,0.005430,0.564849,0.533142,-0.146294,0.674619,-0.710388,-0.974398,-0.516187,0.787815,-0.619477,-1.650251,-0.277016,0.030830,0.186341,-0.592947,0.741486,-0.419233,-0.216435,-1.281538,0.286993,-1.449170,-0.987833,-1.290240,0.808904,-0.720426,0.142199,1.774595,1.108672,0.549016,-0.836152,-0.041443,0.352467,0.239711,-0.194473,0.993187,-0.394312,0.080135,-1.067705,0.003213,-0.149830,-0.752272,0.808738,-1.555710,1.007264,-0.761132,-0.294438,-0.889686,-0.471790,0.771964,0.486710,0.419008,-0.181754,0.565126,0.022265,-0.795735,0.416009,-0.672571,-0.904832,0.443013,1.973624,-0.587153,-0.011289,0.551889,0.469192,-0.789435,0.156678,-0.029855
9826,0.132159,0.539716,-1.052991,-1.080075,-0.682317,-0.2

# Modeling

In [18]:
#| code-summary: 'Data preparation for modeling'
y_train = df_train_bal['y']
y_test = df_test['y']
pos_label = 1 # check 'y'
average = 'macro' # 'binary' (2-class), 'macro' (N-class)
df_res = pd.DataFrame(columns=['Representation', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1'])

## Naive Bayes

In [19]:
#| code-summary: 'Function to train and evaluate Naive Bayes'
def train_eval_mnb(X_train, y_train, X_test, y_test, representation, pos_label, average):
    '''docstring'''
    grid = GridSearchCV(
        MultinomialNB(),
        param_grid={'alpha': [0.01, 0.1, 0.5, 1, 2, 5]},
        cv=5,
        scoring=make_scorer(f1_score, pos_label=pos_label, average=average)
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    
    # Evaluate on held-out test data
    y_pred = best_model.predict(X_test)
    metrics = [
        round(accuracy_score(y_test, y_pred), 2),
        round(precision_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(recall_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(f1_score(y_test, y_pred, pos_label=pos_label, average=average), 2)
    ]
    return [representation, 'Naive Bayes'] + metrics

def train_eval_gnb(X_train, y_train, X_test, y_test, representation, pos_label, average):
    '''docstring'''
    grid = GridSearchCV(
        GaussianNB(),
        param_grid={'var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]},
        cv=5,
        scoring=make_scorer(f1_score, pos_label=pos_label, average=average)
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    
    # Evaluate on held-out test data
    y_pred = best_model.predict(X_test)
    metrics = [
        round(accuracy_score(y_test, y_pred), 2),
        round(precision_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(recall_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(f1_score(y_test, y_pred, pos_label=pos_label, average=average), 2)
    ]
    return [representation, 'Naive Bayes'] + metrics

In [20]:
#| code-summary: 'Run for all representations and report in DataFrame'
df_res.loc[len(df_res)] = train_eval_mnb(X_train_bow, y_train, X_test_bow, y_test, 'Bag of Words', pos_label, average)
df_res.loc[len(df_res)] = train_eval_mnb(X_train_tfidf, y_train, X_test_tfidf, y_test, 'TF-IDF', pos_label, average)
df_res.loc[len(df_res)] = train_eval_gnb(X_train_mean_wv, y_train, X_test_mean_wv, y_test, 'Word2Vec', pos_label, average)
df_res.loc[len(df_res)] = train_eval_gnb(X_train_dv, y_train, X_test_dv, y_test, 'Doc2Vec', pos_label, average)
df_res

c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\utils\validation.py", line 80, in inner_f
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\metrics\_scorer.py", line 328, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

,Representation,Model,Accuracy,Precision,Recall,F1
0,Bag of Words,Naive Bayes,0.91,0.92,0.91,0.91
1,TF-IDF,Naive Bayes,0.91,0.92,0.91,0.91
2,Word2Vec,Naive Bayes,0.88,0.88,0.88,0.88
3,Doc2Vec,Naive Bayes,0.62,0.62,0.62,0.62


## Logistic Regression

In [21]:
#| code-summary: 'Function to train and evaluate Logistic Regression'
def train_eval_logr(X_train, y_train, X_test, y_test, representation, pos_label, average):
    '''docstring'''
    grid = GridSearchCV(
        LogisticRegression(max_iter=1000),
        param_grid={'C': [0.01, 0.1, 1, 10, 100]},
        cv=5,
        scoring=make_scorer(f1_score, pos_label=pos_label, average=average)
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    
    # Evaluate on held-out test data
    y_pred = best_model.predict(X_test)
    metrics = [
        round(accuracy_score(y_test, y_pred), 2),
        round(precision_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(recall_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(f1_score(y_test, y_pred, pos_label=pos_label, average=average), 2)
    ]
    return [representation, 'Logistic Regression'] + metrics

In [22]:
#| code-summary: 'Run for all representations and report in DataFrame'
df_res.loc[len(df_res)] = train_eval_logr(X_train_bow, y_train, X_test_bow, y_test, 'Bag of Words', pos_label, average)
df_res.loc[len(df_res)] = train_eval_logr(X_train_tfidf, y_train, X_test_tfidf, y_test, 'TF-IDF', pos_label, average)
df_res.loc[len(df_res)] = train_eval_logr(X_train_mean_wv, y_train, X_test_mean_wv, y_test, 'Word2Vec', pos_label, average)
df_res.loc[len(df_res)] = train_eval_logr(X_train_dv, y_train, X_test_dv, y_test, 'Doc2Vec', pos_label, average)
df_res

c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\utils\validation.py", line 80, in inner_f
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\metrics\_scorer.py", line 328, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

,Representation,Model,Accuracy,Precision,Recall,F1
0,Bag of Words,Naive Bayes,0.91,0.92,0.91,0.91
1,TF-IDF,Naive Bayes,0.91,0.92,0.91,0.91
2,Word2Vec,Naive Bayes,0.88,0.88,0.88,0.88
3,Doc2Vec,Naive Bayes,0.62,0.62,0.62,0.62
4,Bag of Words,Logistic Regression,0.97,0.97,0.97,0.97
5,TF-IDF,Logistic Regression,0.94,0.94,0.94,0.94
6,Word2Vec,Logistic Regression,0.83,0.83,0.83,0.83
7,Doc2Vec,Logistic Regression,0.55,0.65,0.56,0.47


## Linear SVC

In [23]:
#| code-summary: 'Function to train and evaluate linear SVM'
def train_eval_lsvc(X_train, y_train, X_test, y_test, representation, pos_label, average, random_seed):
    '''docstring'''
    grid = GridSearchCV(
        LinearSVC(random_state=random_seed, max_iter=5000),
        param_grid={'C': [0.01, 0.1, 1, 10, 100]},
        cv=5,
        scoring=make_scorer(f1_score, pos_label=pos_label, average=average)
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    
    # Evaluate on held-out test data
    y_pred = best_model.predict(X_test)
    metrics = [
        round(accuracy_score(y_test, y_pred), 2),
        round(precision_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(recall_score(y_test, y_pred, pos_label=pos_label, average=average), 2),
        round(f1_score(y_test, y_pred, pos_label=pos_label, average=average), 2)
    ]
    return [representation, 'Linear SVM'] + metrics

In [24]:
#| code-summary: 'Run for all representations and report in DataFrame'
df_res.loc[len(df_res)] = train_eval_lsvc(X_train_bow, y_train, X_test_bow, y_test, 'Bag of Words', pos_label, average, RANDOM_SEED)
df_res.loc[len(df_res)] = train_eval_lsvc(X_train_tfidf, y_train, X_test_tfidf, y_test, 'TF-IDF', pos_label, average, RANDOM_SEED)
df_res.loc[len(df_res)] = train_eval_lsvc(X_train_mean_wv, y_train, X_test_mean_wv, y_test, 'Word2Vec', pos_label, average, RANDOM_SEED)
df_res.loc[len(df_res)] = train_eval_lsvc(X_train_dv, y_train, X_test_dv, y_test, 'Doc2Vec', pos_label, average, RANDOM_SEED)
df_res

c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\utils\validation.py", line 80, in inner_f
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "c:\dev\hertie\Sem 5\NLP\NLP_labs_personal\.venv\Lib\site-packages\sklearn\metrics\_scorer.py", line 328, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

,Representation,Model,Accuracy,Precision,Recall,F1
0,Bag of Words,Naive Bayes,0.91,0.92,0.91,0.91
1,TF-IDF,Naive Bayes,0.91,0.92,0.91,0.91
2,Word2Vec,Naive Bayes,0.88,0.88,0.88,0.88
3,Doc2Vec,Naive Bayes,0.62,0.62,0.62,0.62
4,Bag of Words,Logistic Regression,0.97,0.97,0.97,0.97
5,TF-IDF,Logistic Regression,0.94,0.94,0.94,0.94
6,Word2Vec,Logistic Regression,0.83,0.83,0.83,0.83
7,Doc2Vec,Logistic Regression,0.55,0.65,0.56,0.47
8,Bag of Words,Linear SVM,0.96,0.96,0.97,0.96
9,TF-IDF,Linear SVM,0.94,0.95,0.94,0.94
